# Project8 v0.4 — 12.000 ảnh, khởi đầu từ BEST sau 50 epoch

**Chuẩn bị lần train kế tiếp, chưa tự train.** Gắn dataset 12k và `BEST_Project8_20261008.pt` vào Input. Notebook nhận đúng bản qua SHA256, nên không chọn nhầm best sau 30 epoch hoặc dataset v0.3.

- Train **9.540 ảnh**; validation **1.754 ảnh**; test **706 ảnh** giữ riêng, không dùng chọn model.
- 8 ID giữ nguyên. ID 7 `bottle` gồm cup + bottle theo yêu cầu; best cũ tên `cup`.
- Fine-tune mới từ best K3 epoch 19; `resume=False`, khởi tạo lại optimizer/scheduler.
- `cls_remap=False` giữ trọng số phân lớp theo ID 0–7 khi đổi tên ID 7.
- Mặc định 40 epoch mới, patience 10, imgsz 1280, MuSGD lr0=0.001. GPU T4 x2: batch 16; một GPU: batch 8.
- Để `START_TRAINING=False` khi chạy kiểm tra. Chỉ đổi True khi Input và GPU đã sẵn sàng.

[Tài liệu train Ultralytics](https://docs.ultralytics.com/modes/train/) · [Fine-tuning và cls_remap](https://docs.ultralytics.com/guides/finetuning-guide/)

In [ ]:
# 1. Cấu hình lần tiếp theo — chạy các cell theo thứ tự
START_TRAINING = False
DATASET_INPUT = ''  # để trống: tự nhận đúng manifest 12k trong /kaggle/input
CHECKPOINT_INPUT = ''  # để trống: chọn file .pt có đúng SHA256 best 50 epoch
GPU_IDS = 'auto'  # 'auto': dùng tối đa 2 GPU; hoặc '0' / '0,1'
EPOCHS = 40
PATIENCE = 10
IMGSZ = 1280
BATCH = None  # 16 cho 2 GPU, 8 cho 1 GPU; đặt số nhỏ hơn nếu OOM
CLOSE_MOSAIC = 5
RUN_ONE_EPOCH_SMOKE = True  # khi bật train: kiểm 1 epoch trên 5% dữ liệu trước full run
OUTPUT_ROOT = '/kaggle/working/BTL-AI-Camera-v04'
OLD_NAMES = ['person', 'table', 'chair', 'laptop', 'cell phone', 'backpack', 'book', 'cup']
NAMES = OLD_NAMES[:-1] + ['bottle']
EXPECTED_IMAGES = {'train': 9540, 'val': 1754, 'test': 706}
EXPECTED_BOXES = {'train': 28443, 'val': 6861, 'test': 2149}
BEST_SHA256 = 'd4244d26e69d78a066944b22ba64f80ea68602f2f994d6cf6bbae7970ff123f9'
MANIFEST_SHA256 = '03a5e12eecb751f69ff81990201e6f4fc026b9a1ef036fbe96f10264608a0955'
CHECKSUMS_SHA256 = 'fe49b905d17f942002510fefc51b48aa0f495ee5e5355912dfabeb5d70a85357'

In [ ]:
# 2. Giữ phiên bản Ultralytics đã dùng tạo checkpoint; không cài requirements CPU local
import subprocess, sys, torch
before_torch = (torch.__version__, torch.version.cuda)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.165'], check=True)
import ultralytics, yaml
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
from pathlib import Path
import csv, json, hashlib, math, shutil, datetime, gc, zipfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
assert ultralytics.__version__ == '8.4.165'
assert before_torch == (torch.__version__, torch.version.cuda), 'Torch/CUDA bị thay đổi; cần reset môi trường.'
print('Ultralytics:', ultralytics.__version__, '| Torch/CUDA:', before_torch)
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 3. Nhận đúng bộ 12k và checkpoint 50 epoch; không dùng best cũ chỉ vì cùng tên
INPUT_ROOT = Path('/kaggle/input')
def file_sha(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def safe_path(base, relative):
    path = Path(relative)
    if path.is_absolute() or '..' in path.parts:
        raise ValueError(f'Đường dẫn không an toàn: {relative}')
    result = (base / path).resolve()
    if not result.is_relative_to(base.resolve()):
        raise ValueError(f'Đường dẫn ra ngoài dataset: {relative}')
    return result

manifests = [Path(DATASET_INPUT) / 'manifest.csv'] if DATASET_INPUT else sorted(INPUT_ROOT.rglob('manifest.csv'))
valid_roots = [p.parent for p in manifests if p.is_file() and file_sha(p) == MANIFEST_SHA256 and (p.parent/'images/train').is_dir()]
if not valid_roots:
    raise FileNotFoundError('Chưa gắn đúng dataset v0.4 12k. Upload project8_v0.4_merged_20261008.zip như Dataset và Add Input; Kaggle cần bung ZIP.')
if len(valid_roots) != 1:
    raise RuntimeError('Có nhiều bản dataset 12k; đặt DATASET_INPUT cho một bản cụ thể.')
SOURCE_DATA = valid_roots[0]
assert file_sha(SOURCE_DATA / 'checksums.json') == CHECKSUMS_SHA256
checksums = json.loads((SOURCE_DATA / 'checksums.json').read_text(encoding='utf-8'))
with (SOURCE_DATA / 'manifest.csv').open(encoding='utf-8-sig', newline='') as stream:
    rows = list(csv.DictReader(stream))
assert len(rows) == 12000 and Counter(r['split'] for r in rows) == EXPECTED_IMAGES
cfg = yaml.safe_load((SOURCE_DATA/'data.yaml').read_text(encoding='utf-8'))
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][i] for i in range(8)]
assert names == NAMES and cfg['nc'] == 8
assert len({r['pixel_sha256'] for r in rows}) == 12000
expected_files = {r[k] for r in rows for k in ('image_path','label_path')}
assert set(checksums) == expected_files and len(checksums) == 24000

weight_paths = [Path(CHECKPOINT_INPUT)] if CHECKPOINT_INPUT else sorted(INPUT_ROOT.rglob('*.pt'))
matched_weights = [p for p in weight_paths if p.is_file() and file_sha(p) == BEST_SHA256]
if not matched_weights:
    raise FileNotFoundError('Thiếu BEST_Project8_20261008.pt của 50 epoch. Upload/gắn file đúng hash; best của 30 epoch không được dùng.')
initial = matched_weights[0]  # các bản còn lại, nếu có, đều cùng byte/hash
model_check = YOLO(str(initial))
assert model_check.task == 'detect' and [model_check.names[i] for i in range(8)] == OLD_NAMES
print('Dataset:', SOURCE_DATA, '| split:', dict(Counter(r['split'] for r in rows)))
print('BEST 50 epoch:', initial, '| SHA256:', file_sha(initial))
print('ID 7: cup -> bottle (cup+bottle), giữ ID và trọng số qua cls_remap=False.')
del model_check; gc.collect()

In [ ]:
# 4. Stage và kiểm SHA256/tọa độ toàn bộ train+val; test không vào thư mục train
WORK = Path('/tmp/project8_v04_12k_runtime')
DATA_DIR = WORK / 'dataset'
DATA_DIR.mkdir(parents=True, exist_ok=True)
train_val = [r for r in rows if r['split'] in {'train','val'}]

def stage_record(row):
    for key, hash_key in [('image_path','image_sha256'),('label_path','label_sha256')]:
        src = safe_path(SOURCE_DATA, row[key]); dst = safe_path(DATA_DIR, row[key])
        assert checksums[row[key]] == row[hash_key]
        assert file_sha(src) == row[hash_key], f'File nguồn bị đổi: {src}'
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.is_file() or file_sha(dst) != row[hash_key]:
            # copyfile dùng đọc/ghi thường để tránh sendfile trên Kaggle Input mount
            with src.open('rb') as read, dst.open('wb') as write:
                shutil.copyfileobj(read, write, length=1024*1024)
        assert file_sha(dst) == row[hash_key], dst
    counts = Counter()
    for line in (DATA_DIR/row['label_path']).read_text(encoding='utf-8').splitlines():
        values = list(map(float, line.split()))
        assert len(values)==5 and all(math.isfinite(v) for v in values)
        cls,x,y,w,h = values
        assert cls.is_integer() and 0<=cls<8 and 0<=x<=1 and 0<=y<=1 and 0<w<=1 and 0<h<=1
        counts[int(cls)] += 1
    assert sum(counts.values()) == int(row['boxes']) and counts
    return counts

class_counts = Counter()
with ThreadPoolExecutor(max_workers=4) as pool:
    for i, counts in enumerate(pool.map(stage_record, train_val), 1):
        class_counts.update(counts)
        if i % 2000 == 0 or i == len(train_val):
            print('Stage/hash/labels:', i, '/', len(train_val), flush=True)
assert sum(class_counts.values()) == 35304
assert not (DATA_DIR/'images/test').exists() and not (DATA_DIR/'labels/test').exists()
actual_files = {p.relative_to(DATA_DIR).as_posix() for folder in ['images','labels'] for p in (DATA_DIR/folder).rglob('*') if p.is_file() and not p.name.endswith('.cache')}
assert actual_files == {r[k] for r in train_val for k in ('image_path','label_path')}, 'Có file dư trong scratch; dùng session mới.'
DATA_YAML = WORK/'training_data.yaml'
DATA_YAML.write_text(yaml.safe_dump({'path':str(DATA_DIR.resolve()), 'train':'images/train', 'val':'images/val', 'nc':8, 'names':dict(enumerate(NAMES))},sort_keys=False),encoding='utf-8')
resolved = check_det_dataset(str(DATA_YAML),autodownload=False)
assert not resolved.get('test')
for split in ['train','val']:
    assert Path(resolved[split]) == (DATA_DIR/'images'/split).resolve()
OUT = Path(OUTPUT_ROOT); OUT.mkdir(parents=True, exist_ok=True)
PREPARED = {'dataset_manifest_sha256':MANIFEST_SHA256,'checkpoint_sha256':BEST_SHA256,'all_input_images':EXPECTED_IMAGES,'training_images':len(train_val),'training_boxes':35304,'classes':NAMES,'class_id_7':'cups+bottles by user request','resume':False,'cls_remap':False}
(OUT/'prepared.json').write_text(json.dumps(PREPARED,indent=2),encoding='utf-8')
print('PREPARED: 9.540 train + 1.754 val; 706 test giữ riêng. Chưa train.')
print('Box train+val:', {NAMES[i]:class_counts[i] for i in range(8)})

In [ ]:
# 5. Xem ảnh và bounding box của từng lớp trước train
from PIL import Image, ImageDraw
from IPython.display import display
canvas = Image.new('RGB',(4*320,2*260),'#eeeeee')
for cls,name in enumerate(NAMES):
    row = next(r for r in train_val if any(int(line.split()[0])==cls for line in (DATA_DIR/r['label_path']).read_text().splitlines()))
    with Image.open(DATA_DIR/row['image_path']) as im:
        im = im.convert('RGB'); im.thumbnail((312,220))
        draw = ImageDraw.Draw(im)
        for line in (DATA_DIR/row['label_path']).read_text().splitlines():
            c,x,y,w,h = map(float,line.split())
            if int(c)==cls:
                draw.rectangle(((x-w/2)*im.width,(y-h/2)*im.height,(x+w/2)*im.width,(y+h/2)*im.height),outline='lime',width=2)
        x0,y0 = (cls%4)*320,(cls//4)*260
        canvas.paste(im,(x0+(320-im.width)//2,y0))
        ImageDraw.Draw(canvas).text((x0+4,y0+228),f'{cls}: {name}',fill='black')
preview = OUT/'dataset_boxes_preview.jpg'; canvas.save(preview,quality=90)
display(canvas)

In [ ]:
# 6. Fine-tune GPU: MuSGD/lr0=.001; giữ numeric head rows khi cup đổi tên bottle
TRAIN_COMPLETED = False
RUN_DIR = None
if not START_TRAINING:
    print('Đã chuẩn bị. Đặt START_TRAINING=True ở cell 1 khi sẵn sàng, rồi chạy lại theo thứ tự.')
else:
    if not torch.cuda.is_available():
        raise RuntimeError('Cần GPU CUDA trong Settings > Accelerator; không fallback sang CPU.')
    count = torch.cuda.device_count()
    ids = list(range(min(count,2))) if GPU_IDS=='auto' else [int(x) for x in GPU_IDS.split(',')]
    assert ids and len(set(ids))==len(ids) and all(0<=x<count for x in ids)
    device = ','.join(map(str,ids))
    batch = BATCH if BATCH is not None else 8*len(ids)
    assert isinstance(batch,int) and batch>=len(ids) and batch%len(ids)==0
    assert isinstance(EPOCHS,int) and EPOCHS>=10 and PATIENCE>0 and 0<=CLOSE_MOSAIC<EPOCHS
    assert file_sha(initial)==BEST_SHA256
    yolo_cli = shutil.which('yolo')
    if not yolo_cli:
        raise RuntimeError('Không tìm thấy yolo CLI; chạy lại cell cài Ultralytics.')
    # CLI subprocess để DDP chạy trong script entrypoint, không chạy DDP trực tiếp trong notebook.
    stamp = datetime.datetime.now(datetime.timezone(datetime.timedelta(hours=7))).strftime('%Y%m%d_%H%M%S')
    run_name = f'project8_v04_12k_from_best50_{stamp}'
    RUNS = OUT/'kaggle_runs'
    train_config = dict(model=str(initial),data=str(DATA_YAML),project=str(RUNS),name=run_name,
                        epochs=EPOCHS,patience=PATIENCE,imgsz=IMGSZ,batch=batch,device=device,
                        workers=2,cache=False,fraction=1.0,optimizer='MuSGD',lr0=0.001,
                        seed=42,deterministic=True,mosaic=0.5,mixup=0.0,cutmix=0.0,
                        close_mosaic=CLOSE_MOSAIC,save=True,save_period=1,resume=False,
                        cls_remap=False,val=True,nms=False,plots=True,exist_ok=False)
    def run_cli(config,filename):
        config_path = OUT/filename
        config_path.write_text(yaml.safe_dump(config,sort_keys=False),encoding='utf-8')
        command = [yolo_cli,'detect','train',f'cfg={config_path}']
        print('Chạy:',command,'| GPUs:',device,'| batch:',batch,flush=True)
        subprocess.run(command,check=True,cwd=str(OUT))
    if RUN_ONE_EPOCH_SMOKE:
        smoke = dict(train_config,epochs=1,fraction=0.05,close_mosaic=0,name=run_name+'_smoke')
        run_cli(smoke,'smoke_config.yaml')
        plots = sorted((RUNS/(run_name+'_smoke')).glob('train_batch*.jpg'))
        if not plots:
            raise RuntimeError('Smoke không tạo train_batch: kiểm tra dataset trước full run.')
        display(Image.open(plots[0]))
    run_cli(train_config,'train_config.yaml')
    RUN_DIR = RUNS/run_name
    best = RUN_DIR/'weights/best.pt'; last = RUN_DIR/'weights/last.pt'
    assert best.is_file() and last.is_file()
    TRAIN_COMPLETED = True
    print('Full run xong. Best:',best,'| SHA256:',file_sha(best))

In [ ]:
# 7. Chấm BEST cũ và BEST mới trên cùng validation 12k; không dùng test
if not TRAIN_COMPLETED:
    print('Chưa có full run; bỏ qua đánh giá.')
else:
    EVAL_PROTOCOL = dict(data=str(DATA_YAML),split='val',imgsz=640,batch=8,device=ids[0],
                         conf=0.001,iou=0.7,nms=True,max_det=300,augment=False,workers=2,plots=True)
    def evaluate_checkpoint(checkpoint,label):
        evaluated = YOLO(str(checkpoint))
        assert evaluated.task=='detect' and len(evaluated.names)==8
        evaluated.model.names = dict(enumerate(NAMES))  # tên báo cáo; không thay numeric ID hoặc tensor
        metrics = evaluated.val(**EVAL_PROTOCOL,project=str(RUN_DIR/'comparison'),name=label,exist_ok=False)
        record = {'checkpoint_sha256':file_sha(checkpoint),'dataset_manifest_sha256':MANIFEST_SHA256,
                  'protocol':EVAL_PROTOCOL,'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map),
                  'per_class':{}}
        for j,cls in enumerate(metrics.box.ap_class_index):
            record['per_class'][NAMES[int(cls)]] = {'AP50':float(metrics.box.ap50[j]),'AP50_95':float(metrics.box.ap[j])}
        (RUN_DIR/f'{label}_val_metrics.json').write_text(json.dumps(record,indent=2),encoding='utf-8')
        del evaluated; gc.collect(); torch.cuda.empty_cache()
        return record
    before = evaluate_checkpoint(initial,'best50_before')
    after = evaluate_checkpoint(best,'best_v04_after')
    print('mAP50 cũ/mới trên CÙNG val mới:',before['mAP50'],after['mAP50'])
    print('mAP50-95 cũ/mới:',before['mAP50_95'],after['mAP50_95'])
    for name in NAMES:
        print(name, before['per_class'].get(name), '->', after['per_class'].get(name))

In [ ]:
# 8. Đóng ZIP kết quả: best/last, CSV, cấu hình, metadata và cả 2 phép chấm
if not TRAIN_COMPLETED:
    print('Chưa train: không có result ZIP.')
else:
    result = dict(PREPARED,status='completed',run_id=RUN_DIR.name,
                  parent_history_epochs=50,full_run_epochs=len(list(csv.DictReader((RUN_DIR/'results.csv').open()))),
                  smoke_epoch_separate=bool(RUN_ONE_EPOCH_SMOKE),best_sha256=file_sha(best),
                  last_sha256=file_sha(last),ultralytics=ultralytics.__version__,torch=torch.__version__)
    (RUN_DIR/'result.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
    sources = {'best.pt':best,'last.pt':last,'results.csv':RUN_DIR/'results.csv',
               'args.yaml':RUN_DIR/'args.yaml','result.json':RUN_DIR/'result.json',
               'best50_before_val_metrics.json':RUN_DIR/'best50_before_val_metrics.json',
               'best_v04_after_val_metrics.json':RUN_DIR/'best_v04_after_val_metrics.json',
               'prepared.json':OUT/'prepared.json','dataset_boxes_preview.jpg':preview}
    assert all(p.is_file() for p in sources.values()), 'Thiếu kết quả; chạy cell đánh giá trước.'
    files = {name:{'sha256':file_sha(p),'bytes':p.stat().st_size} for name,p in sources.items()}
    result_zip = OUT/(RUN_DIR.name+'_result.zip')
    with zipfile.ZipFile(result_zip,'w',zipfile.ZIP_DEFLATED) as archive:
        for name,p in sources.items(): archive.write(p,name)
        archive.writestr('FILES.json',json.dumps({'schema_version':2,'files':files},indent=2))
    with zipfile.ZipFile(result_zip) as archive:
        assert archive.testzip() is None
    print('Result ZIP:',result_zip,'| SHA256:',file_sha(result_zip))
    print('Dùng Save Version để lưu output hoặc tải ZIP về trước khi session kết thúc.')

## Trước khi chạy full train

1. Input phải có **dataset 12k đúng hash** và **best sau 50 epoch đúng hash**. Cell 3 dừng rõ nếu thiếu; không chọn nhầm nguồn cũ.
2. Chọn GPU trong Notebook Settings. Cell train yêu cầu CUDA; batch tự chọn 16 cho T4 x2 hoặc 8 cho một GPU.
3. Chạy các cell kiểm tra với `START_TRAINING=False`, xem preview box. Khi sẵn sàng đặt True và dùng **Save Version → Save & Run All**.
4. Không so trực tiếp mAP validation mới với số 56,727% trên val v0.3; cell 7 chấm best cũ và mới trên cùng val mới.
5. Nhãn nguồn bổ sung còn cần rà soát thiếu nhãn; chưa kiểm hết ảnh gần trùng/cùng cảnh. Test bottle chỉ 11 ảnh, cần bổ sung test độc lập sau này.
6. Nếu CUDA OOM, giảm BATCH trong một run mới. Không tự đổi về CPU hoặc resume run cũ trên dataset mới.

Mặc định không train khi Run All; 1-epoch smoke chỉ chạy khi bật START_TRAINING và RUN_ONE_EPOCH_SMOKE. Smoke có run riêng, full run khởi đầu lại từ best50.